# Fase 5 · publicación verificada
Formulario y agente usan el verificador. El chat usa el mismo flujo final. Agent y LangSmith remotos esperan credenciales.

In [1]:
from pathlib import Path
import os
if Path.cwd().name == 'notebooks': os.chdir('..')

In [2]:
from pedro.catalog import crear_escenario
from pedro.contracts import Solicitud
from pedro.verification import respuesta_local, verificar
from pedro.agent import renderizar, crear_flujo
from pedro.memory import nueva_sesion, Historial
escenario = crear_escenario(Path('storage/notebooks/catalogos'))
solicitud = Solicitud(producto_id='PEC-01-0001', cantidad=5,
    presupuesto_centavos=1_000_000, plazo_max_dias=7)
respuesta = respuesta_local(escenario, solicitud)
print('errores', verificar(respuesta, escenario))
if not verificar(respuesta, escenario): print(renderizar(respuesta))
memoria = Path('storage/notebooks/final.sqlite3')
sesion = nueva_sesion(memoria, 'alumna', escenario.dataset_id)
flujo = crear_flujo(escenario, memoria)
print(flujo.invoke({'input':'lista piezas'}, config={'configurable':{
    'user_id':'alumna','session_id':sesion}}).content[:300])
print([m.type for m in Historial(memoria, 'alumna', sesion).todos()])

errores []
CentroPeças: R$ 7.279,83 para 5 unidades; plazo 2 días.
- CentroPeças (nacional): R$ 7.279,83; cumple [PEC-01-0001-BR-06]
- Litoral Parts (nacional): R$ 1.930,40; descartada por stock, plazo [PEC-01-0001-BR-05]
- Atlantic Repair Parts (internacional): R$ 7.344,92; descartada por plazo [PEC-01-0001-INT-06]
Historial de precios (informativo; no cambia la recomendación):
- CentroPeças: hoy R$ 1.438,92 por unidad; mes anterior R$ 1.189,87 (+21 %); media de 5 meses R$ 1.372,51 (+5 %).
Fuentes: PEC-01-0001-BR-06, PEC-01-0001-BR-05, PEC-01-0001-INT-06
Piezas del escenario (2640; página 1):
- PEC-01-0001 · NEVE-COMP-0001 · Compresor para Neveras NEVE-COMP-0001
- PEC-01-0002 · NEVE-TERM-0002 · Termostato para Neveras NEVE-TERM-0002
- PEC-01-0003 · NEVE-EVAP-0003 · Evaporador para Neveras NEVE-EVAP-0003
- PEC-01-0004 · NEVE-COND-0004 · Condensador p
['human', 'ai']


/tmp/ipykernel_53068/43665694.py:14: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  flujo = crear_flujo(escenario, memoria)


## Comparación de proveedores: pedido mínimo, origen y preferencia
Ejecución real local, sin modelo. Es el núcleo del asistente: para cada oferta se comprueba stock, **pedido mínimo**, plazo y presupuesto, y se muestra si el proveedor es nacional o internacional. Una preferencia guardada del usuario ordena las ofertas válidas, nunca hace ganar a una que no cumple, y la respuesta dice cuánto cuesta respetarla.

In [3]:
from pedro.business import origen_preferido
casos = [('2 unidades: el proveedor más barato con stock exige un pedido mínimo de 5',
          Solicitud(producto_id='PEC-01-0001', cantidad=2)),
         ('5 unidades: ya se alcanza el pedido mínimo',
          Solicitud(producto_id='PEC-01-0001', cantidad=5)),
         ('sin preferencia: gana la más barata',
          Solicitud(producto_id='PEC-11-0001', cantidad=2)),
         ('con la preferencia guardada «proveedor = nacionales»',
          Solicitud(producto_id='PEC-11-0001', cantidad=2,
                    origen_preferido=origen_preferido({'proveedor': 'nacionales'})))]
for titulo, pedido in casos:
    comparacion = respuesta_local(escenario, pedido)
    print('###', titulo, '| errores de verificación:', verificar(comparacion, escenario))
    print(renderizar(comparacion))
    print()

### 2 unidades: el proveedor más barato con stock exige un pedido mínimo de 5 | errores de verificación: []
Atlantic Repair Parts: R$ 3.200,27 para 2 unidades; plazo 35 días.
- Atlantic Repair Parts (internacional): R$ 3.200,27; cumple [PEC-01-0001-INT-06]
- Litoral Parts (nacional): R$ 819,80; descartada por stock, pedido mínimo (5 unidades) [PEC-01-0001-BR-05]
- CentroPeças (nacional): R$ 2.963,07; descartada por pedido mínimo (5 unidades) [PEC-01-0001-BR-06]
Historial de precios (informativo; no cambia la recomendación):
- Atlantic Repair Parts: hoy R$ 1.381,55 por unidad; mes anterior R$ 1.466,10 (-6 %); media de 5 meses R$ 1.454,84 (-5 %).
Fuentes: PEC-01-0001-INT-06, PEC-01-0001-BR-05, PEC-01-0001-BR-06

### 5 unidades: ya se alcanza el pedido mínimo | errores de verificación: []
CentroPeças: R$ 7.279,83 para 5 unidades; plazo 2 días.
- CentroPeças (nacional): R$ 7.279,83; cumple [PEC-01-0001-BR-06]
- Atlantic Repair Parts (internacional): R$ 7.344,92; cumple [PEC-01-0001-INT-06]

## Aviso de precio: función hardcoded, consejo del modelo y verificación
El historial de precios no cambia el ganador, pero genera un aviso cuando el precio de hoy se aparta de su media. La señal, los hechos y el aviso de plantilla son funciones *hardcoded* y se ejecutan aquí de verdad. **Doble de prueba:** el consejo lo devuelve un modelo guionado y el NLI es un clasificador guionado, para ver los tres desenlaces: consejo respaldado, consejo con una cifra inventada y consejo sin respaldo NLI. La medición con el modelo y el NLI reales está en `docs/trazas/precios_consejo_llm.json`.

In [4]:
import json
from langchain_core.runnables import RunnableLambda
from pedro.precios import aconsejar, aviso_de_precio, crear_consejero, hechos_de_precio, senal_precio
cara = respuesta_local(escenario, Solicitud(producto_id='PEC-02-0189', cantidad=5))
for alternativa in cara.alternativas:
    if alternativa.cumple:
        print('señal:', senal_precio(alternativa.oferta).model_dump())
print('hechos:')
for hecho in hechos_de_precio(cara):
    print('  -', hecho)
print('aviso de plantilla:', aviso_de_precio(cara))
class NLIGuionado:
    def __init__(self, etiqueta): self.etiqueta = etiqueta
    def clasificar(self, premisa, hipotesis): return self.etiqueta
bueno = ('El precio de CentroPeças está 11 % por encima de su media de 5 meses. Atlantic Repair Parts '
         'cuesta 1 % más que la recomendada y está 10 % por debajo de su media.')
casos = [('consejo respaldado', bueno, 'entailment'),
         ('cifra inventada', bueno.replace('11 %', '40 %'), 'entailment'),
         ('sin respaldo NLI', bueno + ' Es una gran oportunidad.', 'neutral')]
for titulo, texto, etiqueta in casos:
    consejero = crear_consejero(RunnableLambda(lambda _, t=texto: json.dumps({'consejo': t})))
    resultado = aconsejar(cara, consejero, NLIGuionado(etiqueta))
    print('###', titulo, '->', resultado['fuente'], '| errores:', resultado['errores'])
    print('    se publica:', resultado['texto'])
print(renderizar(cara).splitlines()[-2])

señal: {'proveedor': 'CentroPeças', 'nivel': 'por encima', 'frente_a_media': 11, 'frente_a_mes_anterior': 6, 'en_minimo': False, 'en_maximo': False}
señal: {'proveedor': 'Atlantic Repair Parts', 'nivel': 'por debajo', 'frente_a_media': -10, 'frente_a_mes_anterior': -3, 'en_minimo': True, 'en_maximo': False}
hechos:
  - La oferta recomendada es la de CentroPeças, con un total de R$ 2.182,51 y entrega en 2 días.
  - El precio de CentroPeças está 11 % por encima de su media de 5 meses.
  - El precio de Atlantic Repair Parts está 10 % por debajo de su media de 5 meses y es el más bajo del periodo.
  - La oferta de Atlantic Repair Parts también cumple: cuesta R$ 12,89 más que la recomendada (1 % más) y entrega en 23 días.
aviso de plantilla: El precio de CentroPeças está 11 % por encima de su media de 5 meses. Atlantic Repair Parts cuesta R$ 12,89 más (1 %) y su precio está 10 % por debajo de su media.
### consejo respaldado -> consejo verificado | errores: []
    se publica: El precio de C

La evaluación remota se ejecuta con evals/evaluar_flujo.py. No se reportan métricas sin esa corrida.